# PipelineNet — кастомная нейронная сеть под задачу

Архитектура, разработанная специально под структуру данных:
- Dual GRU encoder для текущего и предыдущего склада (разные масштабы)
- Route embedding + learned scale factor
- Sinusoidal time features для будущих шагов
- Loss = WAPE + |RelBias| — напрямую метрика соревнования

Результат на LB: **0.344** (хуже LightGBM v7 = 0.32883)

In [ ]:
# ── 0. Зависимости ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import gc

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
TARGET_SEQ = 96   # 48 часов истории target
STATUS_SEQ = 16   # 8 часов истории статусов
H          = 8
STATUS_COLS = ['status_1','status_2','status_3','status_4','status_5','status_6']
STATUS_CURRENT = ['status_1','status_2','status_3']
STATUS_PREV    = ['status_4','status_5','status_6']

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU count: {torch.cuda.device_count()}')

In [ ]:
# ── 1. Загрузка данных ────────────────────────────────────────────────────────
BASE = '/kaggle/input/datasets/nikitagavrilenko/main-dataset'
SUB_BASE = '/kaggle/input/datasets/nikitagavrilenko/sample-sub-wb'

train_df = pd.read_parquet(f'{BASE}/train_solo_track.parquet')
test_df  = pd.read_parquet(f'{BASE}/test_solo_track.parquet')
sub_df   = pd.read_csv(f'{SUB_BASE}/sample_submission.csv')

train_df = train_df.sort_values(['route_id', 'timestamp']).reset_index(drop=True)
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
test_df['timestamp']  = pd.to_datetime(test_df['timestamp'])
train_df[STATUS_COLS] = train_df[STATUS_COLS].fillna(0).astype('float32')

# route → integer index (route_id уже 0-999)
all_routes = sorted(train_df['route_id'].unique())
route2idx  = {r: int(r) for r in all_routes}
NUM_ROUTES = len(all_routes)
print(f'Маршрутов: {NUM_ROUTES}')
print(f'Train rows: {len(train_df):,}')

# Глобальная нормализация статусов (раздельно для current/prev из-за разных масштабов)
status_means = train_df[STATUS_COLS].mean().values.astype('float32')
status_stds  = (train_df[STATUS_COLS].std().values + 1).astype('float32')
print('Status means:', dict(zip(STATUS_COLS, status_means.round(0))))

In [ ]:
# ── 2. Route arrays ───────────────────────────────────────────────────────────
def build_route_arrays(df):
    arrays = {}
    route_medians = df.groupby('route_id')['target_1h'].median()
    for rid, grp in df.groupby('route_id'):
        grp = grp.sort_values('timestamp')
        arrays[int(rid)] = {
            'target'       : grp['target_1h'].values.astype('float32'),
            'status'       : grp[STATUS_COLS].values.astype('float32'),
            'hours'        : grp['timestamp'].dt.hour.values.astype('float32'),
            'dows'         : grp['timestamp'].dt.dayofweek.values.astype('float32'),
            'route_median' : float(route_medians[rid]),
        }
    return arrays

print('Строим route arrays...')
route_arrays = build_route_arrays(train_df)
print('Готово')

In [ ]:
# ── 3. Dataset ────────────────────────────────────────────────────────────────
class ShipmentDataset(Dataset):
    def __init__(self, route_arrays, route2idx, target_seq=TARGET_SEQ,
                 status_seq=STATUS_SEQ, h=H, is_train=True):
        self.target_seq   = target_seq
        self.status_seq   = status_seq
        self.h            = h
        self.is_train     = is_train
        self.route_arrays = route_arrays
        self.route2idx    = route2idx
        self.samples      = []  # (route_id, pos)

        min_start = max(target_seq, status_seq)
        for rid, arrays in route_arrays.items():
            n   = len(arrays['target'])
            end = n - h if is_train else n
            for pos in range(min_start, end):
                self.samples.append((rid, pos))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rid, pos = self.samples[idx]
        arrays   = self.route_arrays[rid]

        target_hist  = arrays['target'][pos - self.target_seq : pos]
        status_hist  = arrays['status'][pos - self.status_seq : pos]
        future_hours = arrays['hours'][pos : pos + self.h]
        future_dows  = arrays['dows'][pos : pos + self.h]
        route_idx    = self.route2idx[rid]
        route_med    = arrays['route_median']

        # нормализация target на медиану маршрута
        target_hist_norm = target_hist / (route_med + 1)

        # нормализация статусов
        status_hist_norm = (status_hist - status_means) / status_stds

        # sinusoidal time features для 8 будущих шагов
        hour_sin = np.sin(2 * np.pi * future_hours / 24).astype('float32')
        hour_cos = np.cos(2 * np.pi * future_hours / 24).astype('float32')
        dow_sin  = np.sin(2 * np.pi * future_dows  / 7).astype('float32')
        dow_cos  = np.cos(2 * np.pi * future_dows  / 7).astype('float32')
        time_feat = np.stack([hour_sin, hour_cos, dow_sin, dow_cos], axis=1)  # [H, 4]

        sample = {
            'target_hist'  : torch.tensor(target_hist_norm, dtype=torch.float32),
            'status_hist'  : torch.tensor(status_hist_norm, dtype=torch.float32),
            'time_feat'    : torch.tensor(time_feat,        dtype=torch.float32),
            'route_idx'    : torch.tensor(route_idx,        dtype=torch.long),
            'route_median' : torch.tensor(route_med,        dtype=torch.float32),
        }

        if self.is_train:
            future_target      = arrays['target'][pos : pos + self.h]
            future_target_norm = future_target / (route_med + 1)
            sample['targets']  = torch.tensor(future_target_norm, dtype=torch.float32)

        return sample


print('Создаём датасет...')
train_ds = ShipmentDataset(route_arrays, route2idx, is_train=True)
print(f'Train samples: {len(train_ds):,}')

# num_workers=0, pin_memory=False обязательно — иначе worker crashes
# (CPU-bound __getitem__ с numpy slicing не параллелится нормально)
train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True,
                          num_workers=0, pin_memory=False)

In [ ]:
# ── 4. Модель ─────────────────────────────────────────────────────────────────
class PipelineNet(nn.Module):
    """
    Архитектура под специфику данных:
    - status_1-3 (текущий склад, малый масштаб) и status_4-6 (предыдущий, крупный) — раздельные GRU
    - route_scale (learned) — каждый маршрут имеет свой масштаб отгрузок
    - Softplus в выходе — гарантирует неотрицательность
    - wape_bias_loss — напрямую оптимизирует метрику соревнования
    """
    def __init__(self, num_routes=NUM_ROUTES, target_seq=TARGET_SEQ,
                 status_seq=STATUS_SEQ, embed_dim=32, hidden=256, h=H):
        super().__init__()
        self.h = h

        # Route embedding + learned scale factor
        self.route_embed = nn.Embedding(num_routes, embed_dim)
        self.route_scale = nn.Embedding(num_routes, 1)
        nn.init.ones_(self.route_scale.weight)

        # Target history encoder
        self.target_encoder = nn.Sequential(
            nn.Linear(target_seq, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
        )

        # Текущий склад (status_1-3): маленькие числа, прямой upstream к target
        self.status_current_gru = nn.GRU(
            input_size=3, hidden_size=hidden // 2,
            num_layers=2, batch_first=True, dropout=0.1,
        )
        # Предыдущий склад (status_4-6): крупный масштаб, leading indicators
        self.status_prev_gru = nn.GRU(
            input_size=3, hidden_size=hidden // 4,
            num_layers=1, batch_first=True,
        )

        # Time features encoder
        self.time_encoder = nn.Sequential(
            nn.Linear(4 * h, hidden // 2),
            nn.GELU(),
            nn.Linear(hidden // 2, hidden // 2),
        )

        # Fusion: target_enc + status_current + status_prev + time + route_emb
        fusion_in = hidden + hidden // 2 + hidden // 4 + hidden // 2 + embed_dim
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Linear(hidden // 2, h),
            nn.Softplus(),  # гарантирует неотрицательность предсказаний
        )

    def forward(self, target_hist, status_hist, time_feat, route_idx):
        # target_hist:  [B, target_seq]
        # status_hist:  [B, status_seq, 6]
        # time_feat:    [B, H, 4]
        # route_idx:    [B]

        t_enc = self.target_encoder(target_hist)          # [B, hidden]

        sc = status_hist[:, :, :3]                        # current: status_1-3
        sp = status_hist[:, :, 3:]                        # prev:    status_4-6
        _, sc_h = self.status_current_gru(sc)
        sc_enc  = sc_h[-1]                                # [B, hidden//2]
        _, sp_h = self.status_prev_gru(sp)
        sp_enc  = sp_h[-1]                                # [B, hidden//4]

        tf       = time_feat.reshape(time_feat.size(0), -1)  # [B, H*4]
        time_enc = self.time_encoder(tf)                  # [B, hidden//2]

        r_emb   = self.route_embed(route_idx)             # [B, embed_dim]
        r_scale = self.route_scale(route_idx).squeeze(-1) # [B]

        fused = torch.cat([t_enc, sc_enc, sp_enc, time_enc, r_emb], dim=-1)
        out   = self.fusion(fused)                        # [B, H] — нормализованный

        return out * r_scale.unsqueeze(-1)                # денормализуем по маршруту


model = PipelineNet(num_routes=NUM_ROUTES).to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f'DataParallel: {torch.cuda.device_count()} GPU')

total_params = sum(p.numel() for p in model.parameters())
print(f'Параметров модели: {total_params:,}')

In [ ]:
# ── 5. Loss: WAPE + |RelBias| — прямо метрика соревнования ───────────────────
def wape_bias_loss(pred, target, eps=1.0):
    wape     = torch.abs(pred - target).sum() / (target.sum() + eps)
    rel_bias = torch.abs(pred.sum() - target.sum()) / (target.sum() + eps)
    return wape + rel_bias

In [ ]:
# ── 6. Обучение ───────────────────────────────────────────────────────────────
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
N_EPOCHS  = 20
scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS * len(train_loader), eta_min=1e-5)

for epoch in range(N_EPOCHS):
    model.train()
    total_loss, n_batches = 0.0, 0

    for batch in train_loader:
        target_hist  = batch['target_hist'].to(DEVICE)
        status_hist  = batch['status_hist'].to(DEVICE)
        time_feat    = batch['time_feat'].to(DEVICE)
        route_idx    = batch['route_idx'].to(DEVICE)
        targets      = batch['targets'].to(DEVICE)

        pred = model(target_hist, status_hist, time_feat, route_idx)
        loss = wape_bias_loss(pred, targets)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        n_batches  += 1

    print(f'Epoch {epoch+1:02d}/{N_EPOCHS}  loss={total_loss/n_batches:.4f}')

torch.save(model.state_dict(), 'pipelinenet.pt')
print('Модель сохранена: pipelinenet.pt')

In [ ]:
# ── 7. Инференс ───────────────────────────────────────────────────────────────
model.eval()
results = []

with torch.no_grad():
    for rid, arrays in route_arrays.items():
        n = len(arrays['target'])
        if n < max(TARGET_SEQ, STATUS_SEQ):
            # Мало истории — фоллбэк на медиану
            for step in range(1, H + 1):
                results.append({'route_id': rid, 'step': step,
                                'y_pred': arrays['route_median']})
            continue

        target_hist_norm = arrays['target'][-TARGET_SEQ:] / (arrays['route_median'] + 1)
        target_hist = torch.tensor(target_hist_norm, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        status_raw  = arrays['status'][-STATUS_SEQ:]
        status_norm = (status_raw - status_means) / status_stds
        status_hist = torch.tensor(status_norm, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        # time features для 8 будущих шагов
        future_hours = arrays['hours'][-H:] if len(arrays['hours']) >= H else np.zeros(H, dtype='float32')
        future_dows  = arrays['dows'][-H:]  if len(arrays['dows'])  >= H else np.zeros(H, dtype='float32')
        hour_sin = np.sin(2 * np.pi * future_hours / 24).astype('float32')
        hour_cos = np.cos(2 * np.pi * future_hours / 24).astype('float32')
        dow_sin  = np.sin(2 * np.pi * future_dows  / 7).astype('float32')
        dow_cos  = np.cos(2 * np.pi * future_dows  / 7).astype('float32')
        time_feat = torch.tensor(
            np.stack([hour_sin, hour_cos, dow_sin, dow_cos], axis=1),
            dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)

        route_idx = torch.tensor([route2idx[rid]], dtype=torch.long).to(DEVICE)

        pred       = model(target_hist, status_hist, time_feat, route_idx)  # [1, H]
        pred_denorm = (pred.squeeze(0).cpu().numpy() * (arrays['route_median'] + 1)).clip(0)

        for step, val in enumerate(pred_denorm, 1):
            results.append({'route_id': rid, 'step': step, 'y_pred': float(val)})

print(f'Предсказаний: {len(results):,}')

In [ ]:
# ── 8. Мерж с test и сабмит ───────────────────────────────────────────────────
test_df['step'] = test_df.groupby('route_id').cumcount() + 1

pred_df = pd.DataFrame(results)
result  = test_df[['id', 'route_id', 'step']].merge(pred_df, on=['route_id', 'step'], how='left')
result['y_pred'] = result['y_pred'].fillna(0).clip(0)

result[['id', 'y_pred']].sort_values('id').to_csv('submission_pipelinenet.csv', index=False)
print('Готово! Файл: submission_pipelinenet.csv')
print(result['y_pred'].describe())